# Sonata + VGC on Colab — Conda Environment (Matches Official Setup)Uses **condacolab** to install miniconda, then **environment.yml** to recreatethe EXACT same environment as the official Pointcept/Sonata repository.**No more version conflicts.** Python 3.10, PyTorch 2.5.0, CUDA 12.4, spconv-cu124 all locked.> After Cell 1 completes, Colab will prompt to **restart the runtime**.> Click Restart, then continue from Cell 2.

In [ ]:
# ═══════════════════════════════════════# CELL 1 — Install Miniconda + Create Env# ═══════════════════════════════════════import subprocess, os, sys# 1. Install condacolab (gets miniconda on Colab)!pip install -q condacolabimport condacolabcondacolab.install()# 2. Downgrade Python env to match official (3.10)#    After condacolab.install(), we have a conda base with python available!conda install -q -y python=3.10# 3. Clone repo (need environment.yml)!rm -rf /content/voxel_group_classifier!git clone --depth 1 https://github.com/yuhang-wang-xjtu/voxel_group_classifier.git /content/voxel_group_classifier# 4. Create conda env from official environment.yml#    This installs: python=3.10, pytorch=2.5.0, cuda=12.4, spconv-cu124, torch-cluster, etc.%cd /content/voxel_group_classifier!conda env create -f environment.yml# 5. Register the env as a Colab kernel!conda run -n pointcept-torch2.5.0-cu12.4 pip install ipykernel!python -m ipykernel install --user --name=pointcept --display-name="Sonata (Python 3.10)"print("\n" + "="*60)print("Environment created. Restart the runtime NOW.")print("After restart, select kernel: 'Sonata (Python 3.10)' from Runtime → Change runtime type")print("Then continue with Cell 2.")

In [ ]:
# ═══════════════════════════════════════# CELL 2 — Verify Environment# ═══════════════════════════════════════# Run AFTER restart — make sure kernel is 'Sonata (Python 3.10)'import torch, spconvprint(f"PyTorch: {torch.__version__}")print(f"CUDA available: {torch.cuda.is_available()}")print(f"CUDA version: {torch.version.cuda}")print(f"spconv: {spconv.__version__}")# Test spconv workstest_conv = spconv.SubMConv3d(3, 3, 3, bias=False).cuda()x = torch.rand(1, 3, 64, 64, 64).cuda()y = test_conv(x)print(f"spconv forward: {y.shape}  ✅")# Compile custom CUDA ops (one-time, as in official setup)%cd /content/voxel_group_classifier!cd libs/pointops && pip install . --quiet 2>&1 | tail -2!cd libs/pointops2 && pip install . --quiet 2>&1 | tail -2!cd libs/pointgroup_ops && pip install . --quiet 2>&1 | tail -2!cd /content/voxel_group_classifier# Check GPU!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ═══════════════════════════════════════# CELL 3 — Download S3DIS + Convert# ═══════════════════════════════════════import numpy as np, h5py, glob, os, urllib.request, zipfileHDF5_URL = "https://huggingface.co/datasets/cminst/S3DIS/resolve/main/indoor3d_sem_seg_hdf5_data.zip"HDF5_DIR = "./indoor3d_sem_seg_hdf5_data"OUT_ROOT = "./data/s3dis"if not os.path.isdir(HDF5_DIR):    print("Downloading S3DIS (~1.7 GB)...")    urllib.request.urlretrieve(HDF5_URL, "/tmp/s3dis.zip")    with zipfile.ZipFile("/tmp/s3dis.zip") as zf:        zf.extractall(HDF5_DIR)h5_files = sorted(glob.glob(f"{HDF5_DIR}/*/*.h5"))print(f"Converting {len(h5_files)} HDF5 files...")for h5_path in h5_files:    fname = os.path.basename(h5_path).replace(".h5", "")    area_idx = int(fname.split("_")[-1])    area = f"Area_{area_idx}"    with h5py.File(h5_path, "r") as f:        data = f["data"][:]        labels = f["label"][:]    xyz = data[..., :3].reshape(-1, 3).astype(np.float32)    rgb = data[..., 3:6].reshape(-1, 3).astype(np.uint8)    seg = labels.reshape(-1, 1).astype(np.int16)    ins = np.zeros_like(seg, dtype=np.int16)    room_dir = os.path.join(OUT_ROOT, area, fname)    os.makedirs(room_dir, exist_ok=True)    np.save(os.path.join(room_dir, "coord.npy"), xyz)    np.save(os.path.join(room_dir, "color.npy"), rgb)    np.save(os.path.join(room_dir, "segment.npy"), seg)    np.save(os.path.join(room_dir, "instance.npy"), ins)print(f"Done. Data at {OUT_ROOT}/")

In [ ]:
# ═══════════════════════════════════════# CELL 4 — Train Sonata + Cosine Masking# ═══════════════════════════════════════MASKING_MODE = "cosine"  # "random" | "cosine" | "vgc"EPOCHS = 10if MASKING_MODE == "cosine":    CONFIG = "configs/sonata/pretrain-sonata-v1m2-cosine-smoketest.py"else:    CONFIG = "configs/sonata/pretrain-sonata-v1m2-vgc-smoketest.py"SAVE = f"exp/{MASKING_MODE}_conda_{EPOCHS}ep"print(f"Config: {CONFIG} | Save: {SAVE}")!python tools/train.py --config-file {CONFIG} --options save_path={SAVE} epoch={EPOCHS} data.train.datasets.0.data_root=data/s3dis enable_wandb=False

In [ ]:
# ═══════════════════════════════════════# CELL 5 — Monitor with TensorBoard# ═══════════════════════════════════════%load_ext tensorboard%tensorboard --logdir exp --port 6006